In [3]:
import torch
import torch.nn as nn

In [4]:
model = nn.Linear(4, 4)

In [5]:
print("Gradients for model weights:", model.weight.grad)

Gradients for model weights: None


In [6]:
def activation(x):
    return torch.where(x < 0, float('-inf'), x)

In [35]:
X = torch.randn(1, 4)
Y = torch.tensor([1])  # Dummy target class index for loss calculation

In [36]:
z = model(X)
a = activation(z)
p = torch.softmax(a, dim=-1)
loss = -torch.log(p[0, Y] + 1e-10)  # Adding a small value to prevent log(0)

z.retain_grad()
a.retain_grad()
p.retain_grad()
loss.retain_grad()

print("Before activation:", z)
print("After activation:", a)
print("Proba:", p)
print("Loss:", loss)

Before activation: tensor([[-0.0546,  0.0948, -0.2367,  1.6178]], grad_fn=<AddmmBackward0>)
After activation: tensor([[  -inf, 0.0948,   -inf, 1.6178]], grad_fn=<WhereBackward0>)
Proba: tensor([[0.0000, 0.1790, 0.0000, 0.8210]], grad_fn=<SoftmaxBackward0>)
Loss: tensor([1.7203], grad_fn=<NegBackward0>)


In [37]:
loss.backward()

In [38]:
print("dL/dL:", loss.grad)
print("dL/dp:", p.grad)
print("dL/da:", a.grad)
print("dL/dz:", z.grad)
print("dL/dweight:", model.weight.grad)

dL/dL: tensor([1.])
dL/dp: tensor([[ 0.0000, -5.5860,  0.0000,  0.0000]])
dL/da: tensor([[ 0.0000, -0.8210,  0.0000,  0.8210]])
dL/dz: tensor([[ 0.0000, -0.8210,  0.0000,  0.8210]])
dL/dweight: tensor([[ 0.0000,  0.0000,  0.0000, -0.0000],
        [-1.0043,  1.7162, -0.2877,  0.3556],
        [ 0.0000,  0.0000,  0.0000, -0.0000],
        [ 1.0043, -1.7162,  0.2877, -0.3556]])


In [42]:
dl_dl = 1.0

dl_dp = -1/p[0, 1]
print("Manual dL/dp:", dl_dp)


dl_da = dl_dp * (p * (1 - p))  # Derivative of softmax
print("Manual dL/da:", dl_da)



Manual dL/dp: tensor(-5.5860, grad_fn=<MulBackward0>)
Manual dL/da: tensor([[-0.0000, -0.8210, -0.0000, -0.8210]], grad_fn=<MulBackward0>)
